# Pitcher Similarity Collaborative Filtering

### Alexander Tu, Dhillon Murphy, Dylan Weinmann

All data from Baseball Savant (https://baseballsavant.mlb.com/)

In [166]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import cdist

In [167]:
raw_pitcher_data = pd.read_csv('data/stats.csv')
raw_pitcher_data.rename(columns={'last_name, first_name': 'name'}, inplace=True)
raw_pitcher_data.columns = [c.split('_')[1] if c.startswith('n_') and c.endswith('_formatted') else c for c in raw_pitcher_data.columns]
raw_pitcher_data.head()

,name,player_id,year,ff,sl,ch,cu,si,fc,fs,kn,st,sv,fo
0,"Cantillo, Joey",676282,2025,42.0,8.7,30.5,18.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"Harris, Hogan",663687,2025,56.7,1.6,13.2,22.8,NaN,2.7,NaN,NaN,3.1,NaN,NaN
2,"Okert, Steven",595345,2025,41.2,57.8,0.1,NaN,NaN,0.9,NaN,NaN,NaN,NaN,NaN
3,"Díaz, Edwin",621242,2025,52.6,47.3,0.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"Irvin, Jake",663623,2025,32.0,4.3,7.7,29.6,22.2,4.2,NaN,NaN,NaN,NaN,NaN


All curveballs are condensed into one 'Curveball' and Forkballs are classified as 'Splitter' due to similar function and lack of Forkball data

In [168]:
pt_dict = {'ff': '4-Seamer',
           'sl': 'Slider',
           'ch': 'Changeup',
           'si': 'Sinker',
           'fc': 'Cutter',
           'st': 'Sweeper',
           }

In [169]:
pitchers = raw_pitcher_data['name'].reset_index(drop=True)

pitch_percentages = (
    raw_pitcher_data[['ff', 'sl', 'ch', 'cu', 'si', 'fc', 'fs', 'kn', 'st', 'sv', 'fo']]
    .copy()
    .fillna(0)
    .assign(
        Curveball=lambda d: d['cu'] + d['kn'] + d['sv'],
        Splitter=lambda d: d['fs'] + d['fo']
    )
    .drop(columns=['cu', 'kn', 'sv', 'fs', 'fo'])
    .rename(columns=pt_dict)
    .reset_index(drop=True)
)
pitch_percentages.head()

,4-Seamer,Slider,Changeup,Sinker,Cutter,Sweeper,Curveball,Splitter
0,42.0,8.7,30.5,0.0,0.0,0.0,18.8,0.0
1,56.7,1.6,13.2,0.0,2.7,3.1,22.8,0.0
2,41.2,57.8,0.1,0.0,0.9,0.0,0.0,0.0
3,52.6,47.3,0.1,0.0,0.0,0.0,0.0,0.0
4,32.0,4.3,7.7,22.2,4.2,0.0,29.6,0.0


In [ ]:
# Uses Centered Log Ratio on the items due to them being raw percentages
# No additional scaling because CLR is a form of scaling already
def clr(x):
    x = x + 0.001
    log_x = np.log(x)
    return log_x - log_x.mean(axis=1, keepdims=True)

In [ ]:
pitcher_clr = clr(pitch_percentages.values)

# Uses L2 norm to compute similarity
dist_matrix = cdist(pitcher_clr, pitcher_clr, metric='euclidean')
sim_matrix = 1 / (1 + dist_matrix)

In [189]:
pitcher_clr

array([[ 5.65997095,  4.08571549,  5.34003699, ..., -4.98547775,
         4.85618758, -4.98547775],
       [ 4.22212267,  0.65495926,  2.76462341, ...,  1.31605546,
         3.31113522, -6.72342446],
       [ 6.50001719,  6.83856074,  0.4889199 , ..., -4.12620062,
        -4.12620062, -4.12620062],
       ...,
       [ 0.12390763, -1.05185294, -1.12328637, ..., -0.34178661,
         0.66197294, -0.59063477],
       [ 3.92323109,  3.25839826,  2.25008579, ...,  2.17689132,
         2.790992  , -6.7978533 ],
       [ 3.17842442,  2.9225224 , -5.95945278, ...,  4.09249788,
        -5.95945278,  2.74022862]], shape=(339, 8))

In [ ]:
# For a given pitcher, prints out k most similar pitchers and their pitch mix
def similar_pitchers(name, k=5):
    i = pitchers[pitchers == name].index[0]
    scores = sim_matrix[i]
    top_k = np.argsort(scores)[::-1][1:k+1]
    print(f'Pitcher: {pitchers.iloc[i]}')
    print(pitch_percentages.iloc[i].to_string())
    print('\n-------\n')
    for num, j in enumerate(top_k):
        print(f'#{num + 1} most similar pitcher: {pitchers.iloc[j]}')
        print(pitch_percentages.iloc[j].to_string())
        print()

In [188]:
similar_pitchers('Jax, Griffin')

Pitcher: Jax, Griffin
4-Seamer     17.7
Slider        0.0
Changeup     22.3
Sinker       11.7
Cutter        4.0
Sweeper      44.0
Curveball     0.2
Splitter      0.0

-------

#1 most similar pitcher: Seymour, Ian
4-Seamer     31.9
Slider        0.0
Changeup     31.7
Sinker        5.4
Cutter       21.5
Sweeper       8.0
Curveball     1.5
Splitter      0.0

#2 most similar pitcher: Holton, Tyler
4-Seamer     11.3
Slider        0.0
Changeup     14.6
Sinker       21.8
Cutter       30.2
Sweeper      18.6
Curveball     3.5
Splitter      0.0

#3 most similar pitcher: Bibee, Tanner
4-Seamer     27.9
Slider        0.0
Changeup     15.2
Sinker       15.2
Cutter       20.9
Sweeper      15.8
Curveball     5.1
Splitter      0.0

#4 most similar pitcher: Pfaadt, Brandon
4-Seamer     23.4
Slider        0.0
Changeup     15.7
Sinker       23.5
Cutter        9.2
Sweeper      18.6
Curveball     9.8
Splitter      0.0

#5 most similar pitcher: Evans, Logan
4-Seamer     11.0
Slider        0.0
Changeup     